# LIFECORE-SWE: Flask Benchmark with gemma-4-31b-it-qat-w4a16-ct

**Google DeepMind Gemma 4 Developer Agent Paper Track — Kaggle 2026**

This notebook runs the LIFECORE-SWE Flask real-repository benchmark using the
competition model `gemma-4-31b-it-qat-w4a16-ct` loaded directly from Kaggle Models.

### Setup required (sidebar):
1. **Models** → Add → search `google/gemma-4` → add `gemma-4-31b-it-qat-w4a16`
2. **Data** → Add → Dataset → search `lifecore-swe` (upload the repo as a dataset first)
3. **Accelerator** → GPU T4 x2 (recommended) or T4 x1
4. Run all cells in order

---
### What this notebook does:
- Installs LIFECORE-SWE from the attached dataset
- Clones Flask v3.0.3 and plants 5 controlled benchmark bugs
- Runs 5 conditions (A=Stateless, B=Static Graph, C=Graph+Text, D=Exp-Graph+Text, E=Exp-Graph-NoText)
- Saves results to `/kaggle/working/results/`
- Prints a summary table with mean±std per condition
- The key finding to verify: **Condition E achieves 100% success with fewer tokens than C**

In [ ]:
# Cell 1: Environment check
import os, sys, pathlib, json, subprocess

# Detect Kaggle environment
IN_KAGGLE = os.environ.get('KAGGLE_KERNEL_RUN_TYPE') is not None or pathlib.Path('/kaggle').exists()
print(f'Running in Kaggle: {IN_KAGGLE}')
print(f'Python: {sys.version}')

# Check GPU
try:
    result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                            capture_output=True, text=True)
    print(f'GPU: {result.stdout.strip()}')
except:
    print('GPU: not detected')

# Check model weights
MODEL_PATHS = [
    '/kaggle/input/gemma4/transformers/gemma-4-31b-it-qat-w4a16/1',
    '/kaggle/input/gemma-4-31b-it-qat-w4a16/1',
    '/kaggle/input/gemma4/transformers/gemma-4-31b-it/1',
    '/kaggle/input/gemma4/transformers/gemma-4-12b-it/1',
]
MODEL_PATH = None
for p in MODEL_PATHS:
    if pathlib.Path(p).exists():
        MODEL_PATH = p
        print(f'Model found: {p}')
        break

if MODEL_PATH is None:
    print('WARNING: No Gemma 4 model weights found in /kaggle/input/')
    print('Add the model via sidebar: Models → google/gemma-4 → gemma-4-31b-it-qat-w4a16')
else:
    contents = list(pathlib.Path(MODEL_PATH).iterdir())[:5]
    print(f'Model dir contents: {[c.name for c in contents]}')

In [ ]:
# Cell 2: Install dependencies
import subprocess

packages = [
    'transformers>=4.40.0',
    'accelerate>=0.28.0',
    'bitsandbytes>=0.43.0',
    'sentencepiece',
    'pytest',
    'flask>=3.0.3',
]

for pkg in packages:
    result = subprocess.run(
        [sys.executable, '-m', 'pip', 'install', pkg, '--quiet'],
        capture_output=True, text=True
    )
    status = 'OK' if result.returncode == 0 else f'FAILED: {result.stderr[-200:]}'
    print(f'  {pkg}: {status}')

print('\nDependencies installed.')

In [ ]:
# Cell 3: Install LIFECORE-SWE from attached dataset or GitHub
import subprocess, sys, pathlib

# Try dataset path first
LIFECORE_PATHS = [
    '/kaggle/input/lifecore-swe',
    '/kaggle/input/lifecore-swe-code',
    '/kaggle/input/lifecoreswe',
]

lifecore_root = None
for p in LIFECORE_PATHS:
    if pathlib.Path(p).exists():
        lifecore_root = p
        print(f'LIFECORE-SWE found at: {p}')
        break

if lifecore_root is None:
    print('Dataset not found in /kaggle/input/. Cloning from GitHub...')
    result = subprocess.run(
        ['git', 'clone', '--depth=1',
         'https://github.com/Innnervision/lifecore-swe.git',
         '/kaggle/working/lifecore-swe'],
        capture_output=True, text=True
    )
    if result.returncode == 0:
        lifecore_root = '/kaggle/working/lifecore-swe'
        print(f'Cloned to: {lifecore_root}')
    else:
        print(f'Clone failed: {result.stderr}')

if lifecore_root:
    sys.path.insert(0, lifecore_root)
    result = subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '-e', lifecore_root, '--quiet'],
        capture_output=True, text=True
    )
    print(f'LIFECORE-SWE install: {"OK" if result.returncode == 0 else result.stderr[-300:]}')

# Quick import test
try:
    from lifecore_swe.agent.kaggle_adapter import KaggleModelAdapter
    from lifecore_swe.codegraph.builder import RepositoryGraphBuilder
    print('Import test: OK')
except Exception as e:
    print(f'Import test FAILED: {e}')

In [ ]:
# Cell 4: Load the model
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

print(f'Loading model from: {MODEL_PATH}')
print(f'GPU memory before load: {torch.cuda.memory_allocated()/1e9:.2f} GB')

# 4-bit QAT quantization config — fits gemma-4-31b on a T4 (16 GB)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    quantization_config=bnb_config,
    device_map='auto',
    torch_dtype=torch.float16,
)
model.eval()

print(f'Model loaded. GPU memory after: {torch.cuda.memory_allocated()/1e9:.2f} GB')
print(f'Model dtype: {next(model.parameters()).dtype}')

In [ ]:
# Cell 5: Build the adapter using the loaded model
import json, re, time
from datetime import datetime, timezone
from typing import Any, Dict, List, Optional
from lifecore_swe.agent.model_adapter import ModelAdapter

class InlineGemmaAdapter(ModelAdapter):
    """Adapter wrapping the already-loaded Gemma 4 model."""

    def __init__(self, model, tokenizer, temperature=0.1, max_new_tokens=512):
        self.model = model
        self.tokenizer = tokenizer
        self.temperature = temperature
        self.max_new_tokens = max_new_tokens
        self.total_input_tokens = 0
        self.total_output_tokens = 0
        self.total_calls = 0
        self.token_log = []

    def get_model_info(self):
        return {
            'model_name': 'gemma-4-31b-it-qat-w4a16-ct',
            'display_name': 'Gemma 4 31B IT QAT W4A16 (Kaggle)',
            'provider': 'Google / Kaggle Models',
            'temperature': self.temperature,
            'is_development_adapter': False,
            'is_competition_model': True,
            'timestamp': datetime.now(timezone.utc).isoformat(),
        }

    def get_token_usage(self):
        return {
            'total_input_tokens': self.total_input_tokens,
            'total_output_tokens': self.total_output_tokens,
            'total_tokens': self.total_input_tokens + self.total_output_tokens,
            'total_calls': self.total_calls,
            'per_call_log': self.token_log,
        }

    def reset_token_usage(self):
        self.total_input_tokens = 0
        self.total_output_tokens = 0
        self.total_calls = 0
        self.token_log = []

    def _generate(self, prompt: str) -> str:
        # Gemma 4 instruction format
        formatted = f'<start_of_turn>user\n{prompt}\n<end_of_turn>\n<start_of_turn>model\n'
        inputs = self.tokenizer(formatted, return_tensors='pt').to(self.model.device)
        in_len = inputs['input_ids'].shape[1]

        with torch.no_grad():
            outputs = self.model.generate(
                **inputs,
                max_new_tokens=self.max_new_tokens,
                do_sample=self.temperature > 0,
                temperature=self.temperature if self.temperature > 0 else None,
                pad_token_id=self.tokenizer.eos_token_id,
            )

        out_len = outputs.shape[1] - in_len
        response = self.tokenizer.decode(outputs[0][in_len:], skip_special_tokens=True).strip()

        self.total_input_tokens += in_len
        self.total_output_tokens += out_len
        self.total_calls += 1
        self.token_log.append({'call_index': self.total_calls,
                                'input_tokens': in_len, 'output_tokens': out_len, 'cached': False})
        return response

    def _parse_json(self, raw: str) -> Any:
        cleaned = raw.strip()
        if cleaned.startswith('```'):
            lines = cleaned.splitlines()[1:]
            if lines and lines[-1].strip() == '```':
                lines = lines[:-1]
            cleaned = '\n'.join(lines).strip()
        try:
            return json.loads(cleaned)
        except Exception:
            m = re.search(r'(\[.*\]|\{.*\})', cleaned, re.DOTALL)
            if m:
                try: return json.loads(m.group(0))
                except: pass
        return {}

    def plan(self, task, context):
        prompt = f'Task:\n{task}\n\nPlan the steps to fix this bug. Output JSON only:\n{{"plan": ["step1","step2","step3"]}}'
        raw = self._generate(prompt)
        parsed = self._parse_json(raw)
        return parsed.get('plan', ['inspect','hypothesize','patch','test','verify'])

    def reason(self, task, context):
        prompt = f'Task:\n{task}\n\nContext:\n{json.dumps(context,indent=2)}\n\nOutput JSON:\n{{"thought_summary":"...","hypothesis":"...","suspected_cause":"...","evidence":["..."],"confidence":0.85,"files_involved":["path.py"]}}'
        raw = self._generate(prompt)
        parsed = self._parse_json(raw)
        if not isinstance(parsed, dict): parsed = {}
        return {'thought_summary': parsed.get('thought_summary',''),
                'hypothesis': parsed.get('hypothesis',''),
                'suspected_cause': parsed.get('suspected_cause',''),
                'evidence': parsed.get('evidence',[]),
                'confidence': float(parsed.get('confidence',0.7)),
                'files_involved': parsed.get('files_involved',[])}

    def select_action(self, state, enable_graph=False):
        graph_block = f"\n=== GRAPH EVIDENCE ===\n{state.get('graph_evidence','')}\n" if state.get('graph_evidence') else ''
        exp_block = f"\n{state.get('experience_context','')}\n" if state.get('experience_context') else ''
        prompt = f"""You are fixing a Python bug. Select ONE next action.

TOOLS: list_files | read_file(path) | search_text(query) | modify_code(path,old_str,new_str) | write_file(path,content) | run_tests | verify | stop(reason)

Task: {state.get('task','')[:500]}
{graph_block}{exp_block}
Phase: {state.get('current_phase','INIT')} | Patch attempts: {state.get('patch_attempts',0)}/3
Files inspected: {state.get('files_inspected',[])}
Last observation: {str(state.get('observations',[''])[-1])[:300]}

Rules: Read file before editing. After tests PASS -> verify. After verify -> stop.

Output ONE JSON object:
{{"thought_summary":"...","hypothesis":"...","confidence":0.8,"next_action":"<action>","tool":"<tool>","arguments":{{}}}}"""
        raw = self._generate(prompt)
        parsed = self._parse_json(raw)
        if not isinstance(parsed, dict): parsed = {}
        action = parsed.get('next_action', parsed.get('action', 'stop'))
        tool = parsed.get('tool', 'filesystem')
        amap = {'replace_in_file':'modify_code','edit_file':'modify_code','patch':'modify_code',
                'verification':'verify','execute_tests':'run_tests','pytest':'run_tests'}
        action = amap.get(action, action)
        if action in ('run_tests','verify') and state.get('current_phase') == 'VERIFY':
            action, tool = 'verify', 'verifier'
        return {'thought_summary': parsed.get('thought_summary',''),
                'hypothesis': parsed.get('hypothesis',''),
                'confidence': float(parsed.get('confidence',0.7)),
                'next_action': action, 'tool': tool,
                'arguments': parsed.get('arguments',{})}

# Instantiate adapter
adapter = InlineGemmaAdapter(model, tokenizer, temperature=0.1, max_new_tokens=512)

# Smoke test
result = adapter.plan('Fix a bug where window_end boundary events are excluded.',
                      {'repo': 'test', 'file': 'filters.py'})
print(f'Adapter smoke test — plan steps: {len(result)}')
print(f'Token usage: {adapter.get_token_usage()["total_input_tokens"]} input tokens')

In [ ]:
# Cell 6: Clone Flask and plant benchmark bugs
import subprocess, pathlib, sys

FLASK_DIR = pathlib.Path('/kaggle/working/real_repo_flask')
TASKS_DIR = FLASK_DIR / 'lifecore_tasks'

if not FLASK_DIR.exists():
    print('Cloning Flask v3.0.3...')
    result = subprocess.run(
        ['git', 'clone', '--depth=1', '--branch', '3.0.3',
         'https://github.com/pallets/flask.git', str(FLASK_DIR)],
        capture_output=True, text=True
    )
    print('Clone:', 'OK' if result.returncode == 0 else result.stderr[-500:])
else:
    print(f'Flask already at {FLASK_DIR}')
    subprocess.run(['git', '-C', str(FLASK_DIR), 'restore', '.'], capture_output=True)

# Install Flask dev dependencies
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e',
                str(FLASK_DIR) + '[dev]', '--quiet'], capture_output=True)
print('Flask installed.')

# Define and plant bugs
TASKS = [
    {'task_id': 'REAL-TASK-1', 'category': 'bug_fix',
     'expected_files': ['src/flask/app.py'],
     'test_target': 'tests/test_basic.py::test_url_mapping',
     'bug': {'file': 'src/flask/app.py',
             'search': '        if provide_automatic_options is None:',
             'replace': '        if provide_automatic_options is not None:'},
     'description': 'Fix URL rule ordering: static route shadowed by variable rule. The condition `provide_automatic_options is None` was accidentally inverted. Restore the original logic.'},
    {'task_id': 'REAL-TASK-2', 'category': 'bug_fix',
     'expected_files': ['src/flask/ctx.py'],
     'test_target': 'tests/test_reqctx.py::test_teardown_request_handler_error',
     'bug': {'file': 'src/flask/ctx.py',
             'search': '            rv = _cv_tokens.get(None)',
             'replace': '            rv = None  # BUG: token lookup disabled'},
     'description': 'Fix RequestContext.pop(): teardown callbacks skipped. The token lookup was disabled. Restore `rv = _cv_tokens.get(None)`.'},
    {'task_id': 'REAL-TASK-3', 'category': 'bug_fix',
     'expected_files': ['src/flask/json/provider.py'],
     'test_target': 'tests/test_basic.py::test_jsonify_basic_types',
     'bug': {'file': 'src/flask/json/provider.py',
             'search': '    mimetype = "application/json"',
             'replace': '    mimetype = "text/plain"  # BUG: wrong mimetype'},
     'description': 'Fix JSON response: mimetype was changed to text/plain. Restore to application/json in DefaultJSONProvider.'},
    {'task_id': 'REAL-TASK-4', 'category': 'behavior_modification',
     'expected_files': ['src/flask/blueprints.py'],
     'test_target': 'tests/test_blueprints.py::test_blueprint_url_prefix',
     'bug': {'file': 'src/flask/blueprints.py',
             'search': '        if url_prefix is not None:',
             'replace': '        if url_prefix is None:  # BUG: inverted condition'},
     'description': 'Fix Blueprint url_prefix: condition was inverted. Restore `if url_prefix is not None:` in BlueprintSetupState.__init__.'},
    {'task_id': 'REAL-TASK-5', 'category': 'bug_fix',
     'expected_files': ['src/flask/sessions.py'],
     'test_target': 'tests/test_basic.py::test_session_cookie_setting',
     'bug': {'file': 'src/flask/sessions.py',
             'search': '        samesite = session_interface.get_cookie_samesite(app)',
             'replace': '        samesite = None  # BUG: samesite always suppressed'},
     'description': 'Fix session cookie SameSite: was suppressed to None. Restore `samesite = session_interface.get_cookie_samesite(app)`.'},
]

def restore_repo():
    subprocess.run(['git', '-C', str(FLASK_DIR), 'restore', '.'],
                   capture_output=True, check=True)

def plant_bug(task):
    patch = task['bug']
    target = FLASK_DIR / patch['file']
    if not target.exists():
        return False
    content = target.read_text(encoding='utf-8')
    if patch['search'] not in content:
        print(f"  WARNING: patch not applicable for {task['task_id']}")
        return False
    target.write_text(content.replace(patch['search'], patch['replace'], 1), encoding='utf-8')
    return True

def confirm_fails(task):
    r = subprocess.run(
        [sys.executable, '-m', 'pytest', task['test_target'], '-x', '-q',
         '--tb=no', '--no-header'],
        cwd=str(FLASK_DIR), capture_output=True, text=True
    )
    return r.returncode != 0

# Write task description files
TASKS_DIR.mkdir(exist_ok=True)
planted = []
for task in TASKS:
    restore_repo()
    ok = plant_bug(task)
    if not ok:
        print(f"  SKIP {task['task_id']}: patch not applicable")
        continue
    fails = confirm_fails(task)
    restore_repo()
    if fails:
        (TASKS_DIR / f"{task['task_id']}.md").write_text(task['description'], encoding='utf-8')
        planted.append(task['task_id'])
        print(f"  PLANTED {task['task_id']}: {task['test_target'].split('::')[-1]}")
    else:
        print(f"  SKIP {task['task_id']}: bug did not break test")

print(f'\nTasks planted: {len(planted)} — {planted}')

In [ ]:
# Cell 7: Core benchmark runner
import time, pathlib, subprocess, json, sys
from lifecore_swe.agent.agent import SWEAgent
from lifecore_swe.experience.models import ExperienceRetrievalConfig
from lifecore_swe.experience.store import ExperienceStore
from lifecore_swe.experience_graph.models import ExperienceGraphConfig
from lifecore_swe.experience_graph.overlay import ExperienceGraphOverlay
from lifecore_swe.retrieval.models import RetrievalConfig
from lifecore_swe.tools.test_runner import TestRunnerTool

RESULTS_DIR = pathlib.Path('/kaggle/working/results')
EXP_DIR     = pathlib.Path('/kaggle/working/experience_graph')
RESULTS_DIR.mkdir(exist_ok=True)
EXP_DIR.mkdir(exist_ok=True)

CONDITIONS = ['STATELESS', 'STATIC_GRAPH', 'GRAPH_EXP_TEXT',
              'EXP_CONDITIONED', 'EXP_GRAPH_NO_TEXT']

def build_agent(task, condition, run_idx=0):
    task_id = task['task_id']
    common = dict(
        repo_path=str(FLASK_DIR),
        task_id=f'REAL-{condition}-{task_id}',
        model_adapter=adapter,
        trajectory_output=str(RESULTS_DIR / f'traj_{condition.lower()}_{task_id.lower()}_r{run_idx}.jsonl'),
        trajectory_mode='M3',
        expected_files=task['expected_files'],
        test_target=task['test_target'],
        task_category=task['category'],
        store_experience_on_completion=True,
        condition=condition,
    )
    store_path   = str(EXP_DIR / f'{condition.lower()}_store.db')
    overlay_path = str(EXP_DIR / f'{condition.lower()}_overlay.db')

    if condition == 'STATELESS':
        return SWEAgent(**common, enable_graph=False, enable_graph_guidance=False,
                        enable_experience=False, enable_experience_graph=False)
    elif condition == 'STATIC_GRAPH':
        return SWEAgent(**common, enable_graph=False, enable_graph_guidance=True,
                        enable_experience=False, enable_experience_graph=False,
                        retrieval_config=RetrievalConfig(max_nodes=15, max_edges=25,
                                                         max_context_chars=4000))
    elif condition == 'GRAPH_EXP_TEXT':
        return SWEAgent(**common, enable_graph=False, enable_graph_guidance=True,
                        enable_experience=True,
                        experience_store=ExperienceStore(db_path=store_path),
                        experience_config=ExperienceRetrievalConfig(max_experiences=3),
                        enable_experience_graph=False,
                        retrieval_config=RetrievalConfig(max_nodes=15, max_edges=25))
    elif condition == 'EXP_CONDITIONED':
        return SWEAgent(**common, enable_graph=False, enable_graph_guidance=False,
                        enable_experience_graph=True,
                        experience_graph_overlay=ExperienceGraphOverlay(db_path=overlay_path),
                        experience_graph_config=ExperienceGraphConfig(max_nodes=15, max_edges=25,
                                                                       enable_experience_text=True),
                        include_raw_experience_text=True, enable_experience=True,
                        experience_store=ExperienceStore(db_path=store_path),
                        experience_config=ExperienceRetrievalConfig(max_experiences=3))
    elif condition == 'EXP_GRAPH_NO_TEXT':
        return SWEAgent(**common, enable_graph=False, enable_graph_guidance=False,
                        enable_experience_graph=True,
                        experience_graph_overlay=ExperienceGraphOverlay(db_path=overlay_path),
                        experience_graph_config=ExperienceGraphConfig(max_nodes=15, max_edges=25,
                                                                       enable_experience_text=False),
                        include_raw_experience_text=False, enable_experience=False,
                        experience_store=ExperienceStore(db_path=store_path))

def run_task(task, condition, run_idx=0):
    task_id = task['task_id']
    print(f'  {task_id} | {condition} | run {run_idx}', end=' ... ')

    restore_repo()
    if not plant_bug(task):
        print('SKIP (patch not applicable)')
        return None
    if not confirm_fails(task):
        restore_repo()
        print('SKIP (bug did not break test)')
        return None

    task_text = (TASKS_DIR / f'{task_id}.md').read_text(encoding='utf-8')
    adapter.reset_token_usage()

    agent = build_agent(task, condition, run_idx)
    result = agent.run(task_description=task_text)
    result.update({'benchmark': 'REAL_REPO_FLASK', 'task_id': task_id,
                   'condition': condition, 'run_index': run_idx,
                   'repository': 'pallets/flask', 'tag': '3.0.3'})

    out = RESULTS_DIR / f'result_{condition.lower()}_{task_id.lower()}_r{run_idx}.json'
    out.write_text(json.dumps(result, indent=2, default=str), encoding='utf-8')

    restore_repo()
    status = result.get('status', '?')
    m = result.get('metrics', {})
    print(f"{status} | calls={m.get('tool_calls','?')} | "
          f"time={m.get('execution_time',0):.0f}s | "
          f"tokens_in={m.get('total_input_tokens',0)}")
    return result

print('Benchmark runner ready.')

In [ ]:
# Cell 8: Run the benchmark (all conditions x all tasks)
# Estimated time: 45-120 min depending on model size and GPU
# To run just one condition: change CONDITIONS_TO_RUN below

CONDITIONS_TO_RUN = CONDITIONS        # all 5
# CONDITIONS_TO_RUN = ['EXP_GRAPH_NO_TEXT', 'STATIC_GRAPH']  # key comparison only
TASKS_TO_RUN = [t for t in TASKS if t['task_id'] in planted]

all_results = []
print('=' * 60)
print(f'Running {len(CONDITIONS_TO_RUN)} conditions x {len(TASKS_TO_RUN)} tasks')
print('=' * 60)

for condition in CONDITIONS_TO_RUN:
    print(f'\nCondition: {condition}')
    for task in TASKS_TO_RUN:
        result = run_task(task, condition, run_idx=0)
        if result:
            all_results.append(result)

print(f'\nTotal runs completed: {len(all_results)}')

In [ ]:
# Cell 9: Summary table and analysis
import statistics
from collections import defaultdict

def agg(vals):
    if not vals: return {'mean': 0, 'std': 0, 'n': 0}
    return {'mean': round(statistics.mean(vals), 2),
            'std': round(statistics.stdev(vals) if len(vals) > 1 else 0.0, 2),
            'n': len(vals)}

cond_data = defaultdict(lambda: defaultdict(list))
for r in all_results:
    c = r.get('condition', '?')
    m = r.get('metrics', {})
    cond_data[c]['success'].append(1 if r.get('status') == 'SUCCESS' else 0)
    cond_data[c]['tool_calls'].append(m.get('tool_calls', 0))
    cond_data[c]['exec_time'].append(m.get('execution_time', 0))
    cond_data[c]['input_tokens'].append(m.get('total_input_tokens', 0))
    cond_data[c]['output_tokens'].append(m.get('total_output_tokens', 0))
    cond_data[c]['searches'].append(m.get('search_count', 0))

print('=' * 80)
print('REAL REPOSITORY BENCHMARK RESULTS (Flask v3.0.3, gemma-4-31b-it-qat-w4a16-ct)')
print('=' * 80)
print(f'{"Condition":<26} {"Success":>8} {"ToolCalls":>11} {"Time(s)":>9} {"InputTok":>10} {"Searches":>9}')
print('-' * 75)
for cond in CONDITIONS:
    if cond not in cond_data: continue
    d = cond_data[cond]
    sr  = agg(d['success'])
    tc  = agg(d['tool_calls'])
    et  = agg(d['exec_time'])
    tok = agg(d['input_tokens'])
    sc  = agg(d['searches'])
    print(f'{cond:<26} {sr["mean"]*100:>7.0f}% '
          f'{tc["mean"]:>8.1f}±{tc["std"]:>4.1f} '
          f'{et["mean"]:>8.0f}s '
          f'{tok["mean"]:>9.0f} '
          f'{sc["mean"]:>8.1f}')
print('-' * 75)

# Save summary
summary = {'benchmark': 'REAL_REPO_FLASK', 'repository': 'pallets/flask',
           'tag': '3.0.3', 'model': 'gemma-4-31b-it-qat-w4a16-ct',
           'conditions': {c: {
               'task_success_rate': agg(cond_data[c]['success']),
               'avg_tool_calls':    agg(cond_data[c]['tool_calls']),
               'avg_execution_time':agg(cond_data[c]['exec_time']),
               'avg_input_tokens':  agg(cond_data[c]['input_tokens']),
               'avg_output_tokens': agg(cond_data[c]['output_tokens']),
               'avg_searches':      agg(cond_data[c]['searches']),
           } for c in cond_data}}

summary_path = RESULTS_DIR / 'real_repo_summary.json'
summary_path.write_text(json.dumps(summary, indent=2), encoding='utf-8')
print(f'\nSummary saved: {summary_path}')
print('\nKey comparison: Condition B (Static Graph) vs Condition E (Exp-Graph No Text)')
for metric, label in [('avg_tool_calls','Tool calls'),('avg_execution_time','Time(s)'),('avg_input_tokens','Input tokens')]:
    b = summary['conditions'].get('STATIC_GRAPH',{}).get(metric,{}).get('mean',0)
    e = summary['conditions'].get('EXP_GRAPH_NO_TEXT',{}).get(metric,{}).get('mean',0)
    if b > 0:
        delta = ((b - e) / b) * 100
        direction = 'E faster/lower' if e < b else 'B faster/lower'
        print(f'  {label:<18}: B={b:.1f}  E={e:.1f}  delta={delta:+.1f}%  ({direction})')

In [ ]:
# Cell 10: Visualize results
try:
    import matplotlib.pyplot as plt
    import numpy as np

    conds = [c for c in CONDITIONS if c in cond_data]
    labels = {'STATELESS':'A\nStateless','STATIC_GRAPH':'B\nStatic\nGraph',
              'GRAPH_EXP_TEXT':'C\nGraph\n+Text','EXP_CONDITIONED':'D\nExp\n+Text',
              'EXP_GRAPH_NO_TEXT':'E\nExp\nNo Text★'}
    colors = ['#EA4335','#4285F4','#FBBC04','#9334E6','#34A853']

    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    fig.suptitle('REAL REPO BENCHMARK: Flask v3.0.3 — gemma-4-31b-it-qat-w4a16-ct',
                 fontsize=12, fontweight='bold')

    for ax, metric, title in [
        (axes[0], 'success',     'Task Success Rate (%)'),
        (axes[1], 'exec_time',   'Avg Execution Time (s)'),
        (axes[2], 'input_tokens','Avg Input Tokens'),
    ]:
        vals = [agg(cond_data[c][metric])['mean'] for c in conds]
        if metric == 'success': vals = [v*100 for v in vals]
        xlabels = [labels.get(c, c) for c in conds]
        bars = ax.bar(xlabels, vals,
                      color=[colors[CONDITIONS.index(c)] for c in conds],
                      edgecolor='white')
        ax.set_title(title)
        for bar, val in zip(bars, vals):
            ax.text(bar.get_x()+bar.get_width()/2, val*1.01,
                    f'{val:.0f}', ha='center', fontsize=8, fontweight='bold')
        if metric == 'success': ax.set_ylim(0, 120)

    plt.tight_layout()
    chart_path = '/kaggle/working/real_repo_benchmark_results.png'
    plt.savefig(chart_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Chart saved: {chart_path}')
except ImportError:
    print('matplotlib not available — skipping chart')

## Results Summary

Copy the numbers from Cell 9 into PAPER.md Section 6 to replace the placeholder text.

Key things to report:
1. **Task success rate** per condition
2. **Input tokens**: Condition C vs Condition E (proves structural memory uses fewer tokens)
3. **Execution time**: Condition B vs Condition E (proves structural conditioning is faster)
4. **Search count**: Should be 0 for B, C, D, E — confirms no search thrashing

After updating the paper, run:
```
python scripts/compute_statistical_analysis.py --source real_repo
```
to add the statistical significance numbers.